# BetaEarth: Generate Embeddings for Any Area

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asterisk-labs/beta-earth/blob/main/examples/generate_demo.ipynb)

This notebook lets you **select an area on an interactive map** and generate dense 10m BetaEarth embeddings for it. Data is downloaded automatically from Microsoft Planetary Computer.

**What you'll get:**
- Annual average embedding (64-band GeoTIFF)
- Per-timestamp embeddings for each S2 / S1 scene
- PCA-RGB preview images for quick visual inspection

## 1. Install dependencies

In [ ]:
!pip install -q betaearth[generate] ipyleaflet

## 2. Select area of interest

Draw a rectangle on the map below. Your bounding box will be captured automatically.

**Tip:** Keep the area small for fast results (< 30 x 30 km). Larger areas work but take longer to download and process.

In [ ]:
from ipyleaflet import Map, DrawControl, basemaps
import ipywidgets as widgets

# Default centre: Bavarian Forest
m = Map(center=(49.0, 13.4), zoom=10, basemap=basemaps.Esri.WorldImagery,
        layout=widgets.Layout(height="500px"))

bbox_store = {}

draw = DrawControl(
    rectangle={"shapeOptions": {"color": "#ff6600", "weight": 2}},
    polygon={}, polyline={}, circle={}, circlemarker={},
)

def on_draw(self, action, geo_json):
    if action == "created":
        coords = geo_json["geometry"]["coordinates"][0]
        lons = [c[0] for c in coords]
        lats = [c[1] for c in coords]
        bbox_store["bbox"] = (min(lons), min(lats), max(lons), max(lats))
        w, s, e, n = bbox_store["bbox"]
        km_w = (e - w) * 111 * abs(np.cos(np.radians((s + n) / 2)))
        km_h = (n - s) * 111
        print(f"Selected bbox: W={w:.4f}, S={s:.4f}, E={e:.4f}, N={n:.4f}")
        print(f"  Size: {km_w:.1f} x {km_h:.1f} km")

draw.on_draw(on_draw)
m.add(draw)

import numpy as np  # needed for on_draw
m

If the interactive map doesn't render (e.g. on GitHub), set the bbox manually:

In [ ]:
# Uncomment and edit to set bbox manually (west, south, east, north):
# bbox_store["bbox"] = (13.18, 48.86, 13.65, 49.13)  # Bavarian Forest

if "bbox" not in bbox_store:
    print("Draw a rectangle on the map above, or set bbox_store['bbox'] manually.")
else:
    print(f"Using bbox: {bbox_store['bbox']}")

## 3. Configure and run

In [ ]:
# --- Settings ---
YEAR = 2023
MIN_COVERAGE = 100    # % — only use scenes that fully cover the AOI
MAX_CLOUD = 20        # % — max cloud cover for S2 scenes
OUTPUT_DIR = "outputs"
DEVICE = "cuda"       # "cpu" if no GPU

assert "bbox" in bbox_store, "Set a bbox first (draw on map or set manually)"
bbox = bbox_store["bbox"]
print(f"Bbox:          {bbox}")
print(f"Year:          {YEAR}")
print(f"Min coverage:  {MIN_COVERAGE}%")
print(f"Max cloud:     {MAX_CLOUD}%")

In [ ]:
from betaearth import BetaEarth
from examples.generate import compute_grid, generate

# Load model
model = BetaEarth.from_pretrained(device=DEVICE)
print(model)

# Compute UTM grid
grid = compute_grid(bbox)
h, w = grid["shape"]
print(f"Grid: {w} x {h} px ({w*10/1000:.1f} x {h*10/1000:.1f} km)")

# Run the full pipeline
from pathlib import Path
output_dir = Path(OUTPUT_DIR) / f"bbox_{bbox[0]:.2f}_{bbox[1]:.2f}"
output_dir.mkdir(parents=True, exist_ok=True)

emb, scenes = generate(
    bbox, YEAR, model, grid, output_dir,
    max_cloud=MAX_CLOUD,
    min_coverage=MIN_COVERAGE,
)

## 4. Visualise results

Display the annual PCA-RGB preview and per-timestamp previews side by side.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

files_dir = output_dir / f"{YEAR}_files"
annual_preview = output_dir / f"{YEAR}_preview_pca.png"

# Collect all preview PNGs
previews = []
if annual_preview.exists():
    previews.append(("Annual average", annual_preview))
for ts_dir in sorted(files_dir.iterdir()):
    png = ts_dir / "preview_pca.png"
    if png.exists():
        previews.append((ts_dir.name, png))

n = len(previews)
cols = min(n, 5)
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.5))
if rows == 1:
    axes = [axes] if n == 1 else list(axes)
else:
    axes = axes.flat

for i, (title, path) in enumerate(previews):
    img = np.array(Image.open(path))
    axes[i].imshow(img)
    axes[i].set_title(title, fontsize=9, fontweight="bold")
    axes[i].axis("off")

for i in range(n, len(axes)):
    axes[i].set_visible(False)

plt.suptitle(f"BetaEarth PCA-RGB Previews — {YEAR}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"\nAll outputs saved to: {output_dir}/")
print(f"  {YEAR}.tif                  — 64-band annual embedding (GeoTIFF)")
print(f"  {YEAR}_preview_pca.png      — PCA-RGB preview")
print(f"  {YEAR}_manifest.json        — scene metadata")
print(f"  {YEAR}_files/*/embedding.tif — per-timestamp embeddings")

## Next steps

- **Download the GeoTIFFs** from the `outputs/` folder in the Colab file browser
- **Open in QGIS** — the 64-band `embedding.tif` files are georeferenced COGs
- **Use from CLI** for larger areas or batch processing:
  ```bash
  pip install betaearth[generate]
  python examples/generate.py --bbox 13.18 48.86 13.65 49.13 --years 2023
  ```
- **Interactive demo** — try the [Streamlit app](../demo/app.py) for browser-based embedding generation